# Phase 10: Natural distribution shift (CIFAR-10.1)

**Internet ON.** Accelerator None is fine (2,000 images). Attach both
`phase-1-sep-14` and `phase-6-sep-15`.

Every result so far uses synthetic corruptions. This is the obvious reviewer
objection. CIFAR-10.1 (Recht et al., "Do CIFAR-10 Classifiers Generalize to
CIFAR-10?", 2018) is a genuinely new 2,000 image test set collected years later
from TinyImages, built specifically to *minimise* distribution shift relative to
the original. Classifiers still lose several points on it.

That makes it the strongest possible test of your claim. If a threshold
calibrated for 5% risk is violated on a test set designed to look like the
training distribution, the failure is not an artifact of synthetic corruption.

Download is 6 MB. Runs in about two minutes.

In [2]:
import glob, os, urllib.request, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision as tv
from scipy.optimize import minimize_scalar
from scipy.stats import beta

SEEDS=[0,1,2]; TG=0.05; ALPHA=0.10; DEV="cuda" if torch.cuda.is_available() else "cpu"
MEAN,STD=(0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)
BASE="https://raw.githubusercontent.com/modestyachts/CIFAR-10.1/master/datasets"
for f in ("cifar10.1_v6_data.npy","cifar10.1_v6_labels.npy"):
    if not os.path.exists(f"/kaggle/working/{f}"):
        urllib.request.urlretrieve(f"{BASE}/{f}", f"/kaggle/working/{f}")
X1=np.load("/kaggle/working/cifar10.1_v6_data.npy")
Y1=np.load("/kaggle/working/cifar10.1_v6_labels.npy").astype(np.int64)
print(X1.shape, X1.dtype, Y1.shape, np.bincount(Y1))

(2000, 32, 32, 3) uint8 (2000,) [200 200 200 200 200 200 200 200 200 200]


## 1. Load both architectures from the attached checkpoints

In [3]:
def resnet18_cifar():
    m=tv.models.resnet18(weights=None,num_classes=10)
    m.conv1=nn.Conv2d(3,64,3,1,1,bias=False); m.maxpool=nn.Identity(); return m
class Block(nn.Module):
    def __init__(s,i,o,st):
        super().__init__(); s.bn1=nn.BatchNorm2d(i); s.c1=nn.Conv2d(i,o,3,st,1,bias=False)
        s.bn2=nn.BatchNorm2d(o); s.c2=nn.Conv2d(o,o,3,1,1,bias=False)
        s.sc=nn.Conv2d(i,o,1,st,bias=False) if (i!=o or st!=1) else None
    def forward(s,x):
        h=F.relu(s.bn1(x)); z=s.c2(F.relu(s.bn2(s.c1(h))))
        return z+(s.sc(h) if s.sc is not None else x)
class WRN(nn.Module):
    def __init__(s,depth=16,k=4,nc=10):
        super().__init__(); n=(depth-4)//6; w=[16,16*k,32*k,64*k]
        s.conv=nn.Conv2d(3,w[0],3,1,1,bias=False); L=[]
        for i in range(3):
            for j in range(n):
                L.append(Block(w[i] if j==0 else w[i+1], w[i+1], (1 if i==0 else 2) if j==0 else 1))
        s.blocks=nn.Sequential(*L); s.bn=nn.BatchNorm2d(w[3]); s.fc=nn.Linear(w[3],nc)
    def forward(s,x):
        z=F.relu(s.bn(s.blocks(s.conv(x))))
        return s.fc(F.adaptive_avg_pool2d(z,1).flatten(1))

def find(pat):
    h=glob.glob(f"/kaggle/input/**/{pat}",recursive=True)
    assert h, f"{pat} not found - attach both notebook outputs"
    return h[0]

MODELS={"res":[],"wrn":[]}
for s in SEEDS:
    m=resnet18_cifar(); m.load_state_dict(torch.load(find(f"ckpt_{s}.pt"),map_location="cpu"))
    MODELS["res"].append(m.to(DEV).eval())
    m=WRN(); m.load_state_dict(torch.load(find(f"wrn_{s}.pt"),map_location="cpu"))
    MODELS["wrn"].append(m.to(DEV).eval())

d1=np.load(find("logits.npz")); d2=np.load(find("logits2.npz")); Y_VAL=d1["y_val"]
print("loaded", DEV)

loaded cpu


## 2. Logits on CIFAR-10.1, same preprocessing as training

In [4]:
MU=torch.tensor(MEAN).view(1,3,1,1); SD=torch.tensor(STD).view(1,3,1,1)
@torch.no_grad()
def logits(m,x):
    o=[]
    for i in range(0,len(x),500):
        b=torch.from_numpy(x[i:i+500]).permute(0,3,1,2).float().div_(255)
        o.append(m(((b-MU)/SD).to(DEV)).float().cpu())
    return torch.cat(o).numpy()

def softmax(z,T=1.0):
    z=z.astype(np.float64)/T; z-=z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def nll(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None)).mean()
def ens(ms,T): return np.mean([softmax(m,T) for m in ms],0)

RAWV={"res":[d1[f"val_{s}"] for s in SEEDS], "wrn":[d2[f"wrn_val_{s}"] for s in SEEDS]}
RAWT={"res":[d1[f"test_{s}"] for s in SEEDS],"wrn":[d2[f"wrn_test_{s}"] for s in SEEDS]}
T_={a:minimize_scalar(lambda t,a=a: nll(ens(RAWV[a],t),Y_VAL),bounds=(.5,10),method="bounded").x
    for a in MODELS}
P101={a:ens([logits(m,X1) for m in MODELS[a]],T_[a]) for a in MODELS}
PVAL={a:ens(RAWV[a],T_[a]) for a in MODELS}
PTEST={a:ens(RAWT[a],T_[a]) for a in MODELS}

print(f"{'arch':>5} {'CIFAR-10 acc':>13} {'CIFAR-10.1 acc':>15} {'drop':>7}")
for a in MODELS:
    a1=(PTEST[a].argmax(1)==d1["y_test"]).mean(); a2=(P101[a].argmax(1)==Y1).mean()
    print(f"{a:>5} {a1*100:12.1f}% {a2*100:14.1f}% {(a1-a2)*100:+6.1f}")

 arch  CIFAR-10 acc  CIFAR-10.1 acc    drop
  res         95.3%           88.9%   +6.4
  wrn         95.0%           88.1%   +6.9


## 3. Does the clean threshold hold on a benign natural shift?

In [5]:
def rc(pr,y,thr):
    a=pr.max(1)>=thr
    if a.sum()==0: return np.nan,0.0
    return 1-(pr.argmax(1)[a]==y[a]).mean(), a.mean()
def thr_for(pr,y,t):
    cf=pr.max(1); o=np.argsort(-cf)
    r=1-np.cumsum(pr.argmax(1)[o]==y[o])/np.arange(1,len(o)+1)
    ok=np.where(r<=t+1e-9)[0]
    return 1.0 if len(ok)==0 else cf[o][ok[-1]]
def cov_for(pr,y,t):
    cf=pr.max(1); o=np.argsort(-cf)
    r=1-np.cumsum(pr.argmax(1)[o]==y[o])/np.arange(1,len(o)+1)
    ok=np.where(r<=t+1e-9)[0]
    return 0.0 if len(ok)==0 else (ok[-1]+1)/len(o)

rng=np.random.default_rng(0)
print(f"{'arch':>5} {'thr':>6} {'risk':>7} {'95% CI':>16} {'ratio':>6} {'cov':>6} {'oracle cov':>11} {'cov err':>8}")
for a in MODELS:
    th=thr_for(PVAL[a],Y_VAL,TG); pr=P101[a]
    r,cv=rc(pr,Y1,th); oc=cov_for(pr,Y1,TG)
    bs=[rc(pr[i],Y1[i],th)[0] for i in (rng.integers(0,len(Y1),len(Y1)) for _ in range(2000))]
    lo,hi=np.nanpercentile(bs,[2.5,97.5])
    print(f"{a:>5} {th:>6.3f} {r:>7.3f} [{lo:.3f}, {hi:.3f}] {r/TG:>5.1f}x {cv:>6.3f} {oc:>11.3f} {cv-oc:>+8.3f}")

 arch    thr    risk           95% CI  ratio    cov  oracle cov  cov err
  res  0.286   0.111 [0.097, 0.125]   2.2x  1.000       0.828   +0.172
  wrn  0.191   0.119 [0.105, 0.133]   2.4x  1.000       0.845   +0.155


## 4. Label free selectors on natural shift

Do ATC and the mean confidence rule behave the same way here as on the synthetic
corruptions?

In [6]:
NE=lambda pr:(pr*np.log(np.clip(pr,1e-12,None))).sum(1)
def _pick(cf,est,t):
    ok=np.where(est<=t+1e-9)[0]
    return 1.0 if len(ok)==0 else np.sort(cf)[::-1][ok[-1]]
print(f"{'arch':>5} {'method':>15} {'risk':>7} {'cov':>6} {'cov err':>8}")
for a in MODELS:
    pr=P101[a]; cv_=PVAL[a].argmax(1)==Y_VAL
    cmc=np.quantile(PVAL[a].max(1),1-cv_.mean()); cne=np.quantile(NE(PVAL[a]),1-cv_.mean())
    cf=pr.max(1); o=np.argsort(-cf); n=np.arange(1,len(o)+1); oc=cov_for(pr,Y1,TG)
    cands={"clean-transfer":thr_for(PVAL[a],Y_VAL,TG),
           "mean-conf":_pick(cf,np.cumsum(1-cf[o])/n,TG),
           "ATC-MC":_pick(cf,np.cumsum(cf[o]<cmc)/n,TG),
           "ATC-NE":_pick(cf,np.cumsum(NE(pr)[o]<cne)/n,TG),
           "oracle":thr_for(pr,Y1,TG)}
    for nm,th in cands.items():
        r,c2=rc(pr,Y1,th); print(f"{a:>5} {nm:>15} {r:>7.3f} {c2:>6.3f} {c2-oc:>+8.3f}")

 arch          method    risk    cov  cov err
  res  clean-transfer   0.111  1.000   +0.172
  res       mean-conf   0.066  0.900   +0.073
  res          ATC-MC   0.086  0.946   +0.118
  res          ATC-NE   0.074  0.915   +0.088
  res          oracle   0.050  0.828   +0.000
  wrn  clean-transfer   0.119  1.000   +0.155
  wrn       mean-conf   0.063  0.879   +0.034
  wrn          ATC-MC   0.095  0.951   +0.106
  wrn          ATC-NE   0.081  0.924   +0.080
  wrn          oracle   0.050  0.845   +0.000


## 5. Label budget on natural shift

Only 2,000 points exist, so budgets are small and the held out set shrinks as n
grows. Fixed sequence testing at full alpha, as in Phase 9.

In [7]:
K=20; ALPHA_C=ALPHA/K
GRIDS={a: np.quantile(PVAL[a].max(1), np.linspace(0.02,0.98,K)) for a in MODELS}
def cp_up(k,n,al): return 1.0 if n==0 else (1.0 if k==n else beta.ppf(1-al,k+1,n-k))
def select(pr,y,idx,a,t=TG):
    cf=pr[idx].max(1); w=pr[idx].argmax(1)!=y[idx]
    for s in GRIDS[a]:
        m=cf>=s
        if m.sum()==0: continue
        if cp_up(w[m].sum(), m.sum(), ALPHA_C)<=t: return s
    return np.inf

print(f"{'arch':>5} {'n':>5} {'risk':>7} {'cov':>7} {'viol':>6} {'abst':>6}")
for a in MODELS:
    pr=P101[a]
    for n in [200,400,800,1200]:
        rs,cs,vs,ab=[],[],[],[]
        for _ in range(200):
            i=rng.permutation(len(Y1)); th=select(pr,Y1,i[:n],a)
            ev=i[n:]; m=pr[ev].max(1)>=th
            if m.sum()==0: rs.append(0.); cs.append(0.); vs.append(False); ab.append(True)
            else:
                r=1-(pr[ev].argmax(1)[m]==Y1[ev][m]).mean()
                rs.append(r); cs.append(m.mean()); vs.append(r>TG); ab.append(False)
        print(f"{a:>5} {n:>5} {np.mean(rs):>7.3f} {np.mean(cs):>7.3f} "
              f"{np.mean(vs)*100:>5.0f}% {np.mean(ab)*100:>5.0f}%")

 arch     n    risk     cov   viol   abst
  res   200   0.007   0.224     0%    64%
  res   400   0.015   0.566     0%     6%
  res   800   0.023   0.687     0%     0%
  res  1200   0.028   0.726     0%     0%
  wrn   200   0.005   0.158     0%    74%
  wrn   400   0.014   0.567     0%     2%
  wrn   800   0.022   0.666     0%     0%
  wrn  1200   0.022   0.680     0%     0%


## What this adds to the paper

If the ratio in section 3 is meaningfully above 1, you can write: the failure
appears even under a natural shift explicitly designed to be minimal, not only
under synthetic corruption. That single sentence removes the main objection to
the whole paper.

If the ratio is close to 1, that is also worth reporting honestly: the failure
requires shift beyond what CIFAR-10.1 represents, which bounds your claim and
tells a reader when they can safely reuse a clean threshold. Either outcome is a
result. Do not rerun it looking for the answer you want.

Caveat to state in the paper: 2,000 images gives wide intervals, which is why
section 3 reports a bootstrap CI.